# Explore Clustering Methodology

This notebook documents the research into the clustering method used in the project.

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans

# 核心聚类逻辑（提取自 src/data/processing.py）
# kmeans = KMeans(n_clusters=10, random_state=42)
# df['cluster'] = kmeans.fit_predict(df[['X', 'Y', 'Z']])

# 验证特征来源：
# X: swiss_easting
# Y: swiss_northing
# Z: dem (数字高程模型)

print('Clustering details identified:')
print('- Algorithm: K-Means')
print('- Number of clusters: 10')
print('- Features: X (Swiss Easting), Y (Swiss Northing), Z (DEM Elevation)')


## Missing Data Construction Processes

This section details how the three missing data patterns (MCAR, SEQ, SCM) are constructed in .

### 共通规则
- **Mask (掩码)**:  表示观测值 (Observed),  表示缺失值 (Missing).
- **Target Rate (**$\pi*)**: 目标整体缺失率。

---

### 1. MCAR (Missing Completely at Random) - 完全随机缺失
最基础的缺失模式，假设观测点的缺失是独立同分布的。
- **构造逻辑**: 对所有站点的时间步执行独立的 Bernoulli 采样。
- **公式**: (s, t) \sim Bernoulli(1 - \pi)$
- **代码实现**: 

---

### 2. SEQ (Sequential Missing) - 连续/序列缺失
模拟传感器故障或通信中断导致的较长时间段连续缺失。
- **构造逻辑**: 每个站点独立生成若干缺失块。
- **缺失块长度**: 服从 (n1, p1)$。
- **观测块长度**: {obse\_base} + Binomial(n0, p0)$。
- **动态调整**:
  - 根据目标缺失率 $\pi$ 计算所需缺失块数量 $。
  - 动态计算 $ 以使最终期望缺失率逼近 $\pi$。
- **Cyclic Shift (循环移位)**: 生成初始 mask 后，在随机位置切分并交换两部分，以消除序列起始处分布不均的偏差。

---

### 3. SCM (Spatially Correlated Missing) - 空间相关缺失
模拟区域性气象事件（如雷暴、区域性断电）导致的地理邻近站点同步缺失。
- **构造逻辑**: 结合了“聚类模板”和“站点特异性擾动”。
- **步骤**:
  1. **聚类模板**: 对于同一个簇中的所有站点，使用**相同的 Seed** 调用  生成基础连续缺失块。
  2. **站点背景噪声**: 为每个站点生成一个独立的 MCAR mask ()，失效率由  (默认 0.95) 控制。
  3. **融合 (OR Operation)**: {final} = M_{base} \lor M_{seq\_template}$。
- **核心特性**: 
  - 只有当  和  同时判定为缺失 (0) 时，最终结果才为缺失。
  - 同簇站点因为共享  的种子，其缺失发生的时段和时长高度一致，但  赋予了它们微小的个体差异。


## Missing Data Construction Processes

This section details how the three missing data patterns (MCAR, SEQ, SCM) are constructed in `src/data/misser.py`.

### 共通规则
- **Mask (掩码)**: `1` 表示观测值 (Observed), `0` 表示缺失值 (Missing).
- **Target Rate ($\pi$)**: 目标整体缺失率。

---

### 1. MCAR (Missing Completely at Random) - 完全随机缺失
最基础的缺失模式，假设观测点的缺失是独立同分布的。
- **构造逻辑**: 对所有站点的时间步执行独立的 Bernoulli 采样。
- **公式**: $M(s, t) \sim Bernoulli(1 - \pi)$
- **代码实现**: `rng.choice([0, 1], size=shape, p=[pi, 1 - pi])`

---

### 2. SEQ (Sequential Missing) - 连续/序列缺失
模拟传感器故障或通信中断导致的较长时间段连续缺失。
- **构造逻辑**: 每个站点独立生成若干缺失块。
- **缺失块长度**: 服从 $Binomial(n1, p1)$。
- **观测块长度**: $L_{obse\_base} + Binomial(n0, p0)$。
- **动态调整**:
  - 根据目标缺失率 $\pi$ 计算所需缺失块数量 $k$。
  - 动态计算 $n0$ 以使最终期望缺失率逼近 $\pi$。
- **Cyclic Shift (循环移位)**: 生成初始 mask 后，在随机位置切分并交换两部分，以消除序列起始处分布不均的偏差。

---

### 3. SCM (Spatially Correlated Missing) - 空间相关缺失
模拟区域性气象事件（如雷暴、区域性断电）导致的地理邻近站点同步缺失。
- **构造逻辑**: 结合了“聚类模板”和“站点特异性扰动”。
- **步骤**:
  1. **聚类模板**: 对于同一个簇中的所有站点，使用**相同的 Seed** 调用 `single_seq_masker` 生成基础连续缺失块。
  2. **站点背景噪声**: 为每个站点生成一个独立的 MCAR mask (`mask_base`)，失效率由 `pi_hat` (默认 0.95) 控制。
  3. **融合 (OR Operation)**: $M_{final} = M_{base} \lor M_{seq\_template}$。
- **核心特性**: 
  - 只有当 `mask_base` 和 `seq_template` 同时判定为缺失 (0) 时，最终结果才为缺失。
  - 同簇站点因为共享 `seq_template` 的种子，其缺失发生的时段和时长高度一致，但 `mask_base` 赋予了它们微小的个体差异。
